In [16]:
import numpy as np
from matplotlib import pyplot as plt
import matplotlib.animation as animation
from matplotlib.animation import FuncAnimation, PillowWriter 
import pandas as pd
import imageio, re
import json, os
import frozensoil as FS
import matplotlib.dates as mdates

In [17]:
dyn = (1.6 - 0.5)/0.5*100
head0 = (1.4 - 0.5)/0.5*100
print (dyn, head0)

220.00000000000003 179.99999999999997


CREATE SURFACE DATA WITH 2 degrees variation around the mean so .. max increase by 2 and min decreases by 2

In [18]:
fs = FS.FrozenSoil(cfg_file='./configs/config_smc.json',smc_file_flag=True,tc_constant=False,standalone_HE=False)

#### Initialize the state of the model at the initial time, sets all the variables needed for phase partition and heat equation

In [19]:
fs.initialize()

#### All set for running the model and advancing the timestep. The forcing data (time series), if provided, will be used and the corresponding variables will be updated each time step at the model advancement stage

In [20]:
#set top surface temperature, if the boundary condition is not set here, the default is -12 degree C
cycles = fs.get_NTsteps()

#cycles = 365*24*3

In [21]:
cycles

52534

In [22]:
fs.run_model_forcing()

#### Postprocessing
#### Solution format: 0: time [s], 1: soil temperature [K], 2: liquid water [-], 3: total soil moisture [-]

In [23]:
solutionA = fs.get_solution()

In [24]:
len(solutionA)

52534

In [25]:
Time = [solutionA[i][0]/86400 for i in range(cycles)]
Time_s = [solutionA[i][0] for i in range(cycles)]
print (len(Time), cycles)


def get_dt(yr=2015):
    Time_ymd = []
    #tn = pd.Timedelta('00:00:00')
    print (len(Time), cycles)
    start = pd.Timestamp(year=yr,month=1,day=1)
    for i in range(cycles):
        #print (i)
        #tt = Data['Time'][1] - Data['Time'][0]
        t1 = pd.to_timedelta(Time_s[i],unit='s') #+ pd.to_timedelta(Time[i])
        t2 = start + t1
        #tn = tn + pd.Timedelta(DataSMCST['Time'][i+1]-DataSMCST['Time'][i])
        Time_ymd.append(start + t1)
        #print (start,t2, Time_s[i])
        #break
        #print (i, t1, t2)
    return Time_ymd
Time_ymd = get_dt(yr=2011)
Time_ymd[-1]

52534 52534
52534 52534


Timestamp('2016-12-31 23:00:00')

In [31]:
#site_name = 'SC2093'
#site_name = 'SC2197'
#site_name = 'SC2120'
site_name = 'SC2039'

infile="/Users/ahmadjan/Core/SimulationsData/forcing/SCAN/%s/SMC_ST/hourly_file_H.csv"%site_name

DataOB = pd.read_csv(infile,sep=',',skiprows=0)#,usecols=range(7))
names = DataOB.columns
names

Index(['TIME', 'SURFBC', 'ST5', 'ST10', 'ST20', 'ST50', 'ST100', 'SMCT'], dtype='object')

In [32]:
def get_RMSE(d_ob,d_sim):
    v1 = np.linalg.norm(d_ob - d_sim) / np.sqrt(len(d_sim))
    return np.round(v1,2)

def get_NSE(d_ob,d_sim):
    ll = len(d_sim)
    v2 = 0
    v3 = 0
    ob_mean = np.mean(d_ob)
    for j in range(ll):
        v2 = v2 +  (d_ob[j] - d_sim[j])**2.0
        v3 = v3 +  (d_ob[j] - ob_mean)**2.0
    return np.round(1-v2/v3,2)

In [33]:
colors = ['r','k','g','m','c','y','b']


In [34]:
Z =[0.05,0.1,0.2,0.3,0.4,0.5,0.6,0.7,0.8,0.9,1.0,1.1,1.2,1.3,1.4,1.5,1.6,1.7,1.8,1.9,2.0,2.1,2.2,2.3,2.4,2.5,2.6,2.7,2.8,2.9,3.0,3.1,3.2,3.3,3.4,3.5,3.6,3.7,3.8,3.9,4.0,4.1,4.2,4.3,4.4,4.5,4.6,4.7,4.8,4.9,5.0,5.1,5.2,5.3,5.4,5.5,5.6,5.7,5.8,5.9,6.0]
Z = [i*100 for i in Z]

In [35]:
%matplotlib qt
#fig, axs = plt.subplots(1,4,figsize=(12,4))
fig, axs = plt.subplots(2,2, figsize=(10,6), facecolor='w', edgecolor='k',gridspec_kw={'width_ratios':[1,1]})
fig.subplots_adjust(hspace =.02, wspace=1.02)
axs = axs.ravel()
xfmt = mdates.DateFormatter('%y/%m')
plt.tight_layout()
depth = [0, 1,5,10]
T1, T2, T3,T4,T5 = [], [], [],[],[]
C_to_K = 273.15
for j in range(cycles):
    T1.append(solutionA[j][1][0] - C_to_K) #0.25 m
    T2.append(solutionA[j][1][1]- C_to_K) # 1.0m
    T3.append(solutionA[j][1][5]- C_to_K) # 2.0 m
    T4.append(solutionA[j][1][10]- C_to_K) # 2.75 m
    #T5.append(solutionA[j][1][depth[4]]) # 3.0 m

#print (len(T1))

ST = [T1, T2, T3, T4]
OB_names = ['ST5', 'ST10','ST50','ST100']

for i in range(4):
    print (names[2+i])
    d1 = np.array(DataOB[OB_names[i]][:cycles]) - C_to_K
    d2 = ST[i]
    if i < 3:
        axs[i].plot(Time_ymd,d1, color=colors[0],linestyle='dotted')
        axs[i].plot(Time_ymd,d2, color=colors[1])    
    else:
        axs[i].plot(Time_ymd,d1, color=colors[0],linestyle='dotted',label='Observed')
        axs[i].plot(Time_ymd,d2, color=colors[1],label='Simulated')
    
    start = 365*24
    rmse = get_RMSE(d1[start:],d2[start:]) # skip 365*24 timesteps (1 year) of the spinup period
    #print (d1)
    nse = get_NSE(d1[start:],d2[start:]) 
    axs[i].text(Time_ymd[500],-12, 'Depth = %s cm'%Z[depth[i]])
    axs[i].text(Time_ymd[500],38,'RMSE, NSE=%s, %s'%(rmse, nse))
    
for i in range(4):
    axs[i].set_ylabel('Soil Temperature [\u00B0C]',fontsize=12)
    axs[2].set_xlabel('Time [y/m]',fontsize=12)
    axs[3].set_xlabel('Time [y/m]',fontsize=12)
    axs[3].legend(loc='upper right')
    axs[i].hlines(y=0, xmin=Time_ymd[0],xmax=Time_ymd[-1], color='grey',linestyle='dashed')
    axs[i].set_xlim(Time_ymd[0],Time_ymd[-1])
    
    axs[i].xaxis.set_major_formatter(xfmt)
    axs[i].set_ylim(-15,42)
#axs[1].set_ylim(270,300)

plt.tight_layout()
outfile='/Users/ahmadjan/Core/SimulationsData/postprocessing/frozensoil/ModelEvaluation/'
#plt.savefig(outfile+'site2093_ST_BC_5cm_bot285.jpg', bbox_inches='tight', dpi=200)
#plt.savefig(outfile+'site2197_ST_BC_5cm_bot280.jpg', bbox_inches='tight', dpi=200)
#plt.savefig(outfile+'site2039_ST_BC_5cm_bot280.jpg', bbox_inches='tight', dpi=200)
plt.savefig(outfile+'%s_ST_BC_5cm_bot280_H.jpg'%site_name, bbox_inches='tight', dpi=200)

ST5
ST10
ST20
ST50


In [ ]:
#solutionA[8651]

In [ ]:
RMSE = []
NSE = []
for i in range(4):
    v1 = np.linalg.norm(DataOB[OB_names[i]][365*24:cycles] - C_to_K - ST[i][365*24:]) / np.sqrt(len(ST[i][365*24::]))
    RMSE.append(v1)
    ll = len(ST[i])
    v2 = 0
    v3 = 0
    ST_ob = np.array(DataOB[OB_names[i]]) - C_to_K
    #print (ST_ob)
    ob_mean = np.mean(ST_ob)
    ST_sim = ST[i]
    for j in range(ll):
        v2 = v2 +  (ST_ob[j] - ST_sim[j])**2.0
        v3 = v3 +  (ST_ob[j] - ob_mean)**2.0
    NSE.append(1-v2/v3)

In [ ]:
NSE
#[0.9578549863832893, 0.9541404379185615, 0.952303659867533, 0.9042251170567273]

In [ ]:
fig, axs = plt.subplots(1,1,figsize=(6,4))
#axs = axs.ravel()
#axs.bar(['5', '10', '50', '100'], RMSE,color='lightgrey',width=0.5,label='RMSE')
#axs.bar(['5', '10', '50', '100'], NSE,color='grey',width=0.25, label='NSE')

axs.bar(np.arange(1,5)-0.125, RMSE,color='lightgrey',width=0.25,label='RMSE')
axs.bar(np.arange(1,5)+0.125, NSE,color='grey',width=0.25,label='NSE')
axs.set_xticks(np.arange(1,5))
axs.set_xticklabels(['5', '10', '50', '100'])
for i in range(1):
    axs.set_ylabel('Error [\u00B0C]',fontsize=12)
    axs.set_xlabel('Depth [cm]',fontsize=12)
    if i ==0:
        axs.set_ylim(0,2)
        axs.set_yticks(np.arange(0, 2.001,1))
    axs.legend()
#plt.savefig(outfile + '/site_2093-RMSE_ST_vs_OB.png',dpi=200)
#plt.savefig(outfile + '/site_2197-RMSE_ST_vs_OB.png',dpi=200)
#plt.savefig(outfile + '/site_2030-RMSE_ST_vs_OB.png',dpi=200)
plt.savefig(outfile + '/%s-RMSE_ST_vs_OB.png'%site_name,dpi=200)
